In [ ]:
import os, h5py, pickle
import numpy as np
import matplotlib.pyplot as plt

from main_functions_generalAPI import *
from scipy.interpolate import interp1d

recording_path = r"data\2026-02-25_mb_fish1_rec2"
save_path = r"results\2026-02-25_mb_fish1_rec2"

In [ ]:
i=0
with open(os.path.join(recording_path, 'recording_dicts', f'recording_neuron{str(i)}.pkl'), 'rb') as recfile:
    recording = pickle.load(recfile)

In [ ]:
list(recording.keys())

In [ ]:
camera=h5py.File(os.path.join(recording_path, 'Camera.hdf5'), 'r')

In [ ]:
list(camera.keys())

In [ ]:
record_grou_id=np.array(camera['__record_group_id'])
plt.plot(record_grou_id)
np.unique(record_grou_id)

In [ ]:
np.array(camera['eyepos_ang_le_pos_0']).squeeze()

In [ ]:
t1, t2=1000,4000
%matplotlib qt
plt.figure(figsize=(20,5))
plt.plot(np.array(camera['fish_embedded_frame_time']).squeeze()[t1:t2],
         np.array(camera['eyepos_ang_le_pos_0']).squeeze()[t1:t2], label='angular')
plt.plot(np.array(camera['fish_embedded_frame_time']).squeeze()[t1:t2],
         np.array(camera['eyepos_le_axes_0']).squeeze()[t1:t2]-np.array(camera['eyepos_le_axes_0']).squeeze()[t1:t2].mean(axis=0), label='absoloute')
plt.legend()

In [ ]:
np.array(camera['eyepos_le_axes_0']).squeeze()[t1:t2].mean(axis=0)

In [ ]:
np.array(camera['eyepos_le_axes_0']).squeeze()

In [ ]:
np.array(camera['__time']).squeeze()

In [ ]:
np.array(camera['fish_embedded_frame_time']).squeeze()

In [ ]:
1/np.diff(np.array(camera['fish_embedded_frame_time']).squeeze()).mean()

In [ ]:
time_camera=np.array(camera['__time']).squeeze()
time_camera.shape

In [ ]:
# camera has ~60Hz
1/np.diff(time_camera).mean()

In [ ]:
recording['ca_times']

In [ ]:
recording['ca_times'][:1] <= time_camera[:,None]

In [ ]:
t1, t2=1000,2000
%matplotlib qt

left=np.array(camera['eyepos_ang_le_pos_0']).squeeze()[t1:t2]
right=np.array(camera['eyepos_ang_re_pos_0']).squeeze()[t1:t2]
time=np.array(camera['fish_embedded_frame_time']).squeeze()[t1:t2]-np.array(camera['fish_embedded_frame_time']).squeeze()[0]
time/=60

plt.figure(figsize=(20,5))
plt.plot(time,
         left, label='LEFT')
plt.plot(time,
         right, label='RIGHT')

plt.legend()

In [ ]:
left_eye=np.array(camera['eyepos_ang_le_pos_0']).squeeze()
right_eye=np.array(camera['eyepos_ang_re_pos_0']).squeeze()
plt.hist(left_eye, bins=60, alpha=0.5)
plt.hist(right_eye, bins=60, alpha=0.5);

In [ ]:
plt.scatter(left_eye, right_eye, s=.5, alpha=0.2)

In [ ]:
time_eye=np.array(camera['fish_embedded_frame_time']).squeeze()
left_eye=np.array(camera['eyepos_ang_le_pos_0']).squeeze()
right_eye=np.array(camera['eyepos_ang_re_pos_0']).squeeze()
#eye_pos_resampled=np.empty(recording['ca_times'].shape +(2,))

# calculate average eye position within each ca time frame
# for i, (timebin_start, timebin_end) in enumerate(zip(recording['ca_times'][:-1], recording['ca_times'][1:])):
#     mask_timebin=np.where(np.logical_and(time_eye>timebin_start, time_eye<timebin_end))
#     eye_pos_resampled[i,0]=left_eye[mask_timebin].mean()
#     eye_pos_resampled[i,1]=right_eye[mask_timebin].mean()


In [ ]:
time_eye.shape, left_eye.shape, right_eye.shape, eye_pos.shape

In [ ]:
def plot_eyepositions(
        eyepos,
        q1_min_left,
        q1_min_right,
        q1_width,
        q1_height,
        q3_max_left,
        q3_max_right,
        q3_widht,
        q3_height):
    """
        Scatterplot of eye positions to visualize quadrants for subselecting data.
    """
    plt.figure(figsize=(20,20))
    plt.scatter(eyepos[:,0], eyepos[:,1], s=1., alpha=0.6)
    plt.xlabel('left')
    plt.ylabel('right')

    # 1st quadrant (defined by lower boundaries)
    plt.hlines((q1_min_right, q1_min_right+q1_height),q1_min_left, q1_min_left+q1_width)
    plt.vlines((q1_min_left, q1_min_left+q1_width),  q1_min_right, q1_min_right+q1_height)

    # 3rd quadrant (define by upper boundaries)
    plt.hlines((q3_max_right, q3_max_right-q3_height), q3_max_left-q3_widht, q3_max_left)
    plt.vlines((q3_max_left, q3_max_left-q3_widht), q3_max_right-q3_height, q3_max_right)


In [ ]:
# Define quadrants
eye_pos=np.column_stack(
    (np.array(camera['eyepos_ang_le_pos_0']).squeeze(),
     np.array(camera['eyepos_ang_re_pos_0']).squeeze()))



In [ ]:
q1_min_left=10
q1_min_right=-5
q3_max_left=6
q3_max_right=-12

# 1st quadrant (defined by lower boundaries)
# 3rd quadrant (define by upper boundaries)
plot_eyepositions(eye_pos,
                  q1_min_left=q1_min_left,
                  q1_min_right=q1_min_right,
                  q1_width=15,
                  q1_height=28,
                  q3_max_left=q3_max_left,
                  q3_max_right=q3_max_right,
                  q3_widht=20,
                  q3_height=20)

# quadrant masks to apply to motion vectors
q1_mask=np.logical_and(eye_pos[:,0] > q1_min_left,eye_pos[:,1] > q1_min_right)
q3_mask=np.logical_and(eye_pos[:,0] < q3_max_left, eye_pos[:,1] < q3_max_right)
q1_mask.shape, q3_mask.shape

In [ ]:
motion_vectors_2d = recording['cmn_motion_vectors_2d']

In [ ]:
Dff_all_neuron = np.load(os.path.join(save_path, "deconvolved_Dff_original.npy"))
Dff_resampled = interp1d(recording['ca_times'], Dff_all_neuron, kind='nearest')(
        recording['time_resampled'])

In [ ]:
# original signal is recorded with ~2Hz
# resampled signal is upsampled and interpolated to match Display frequency for ETA calculation
Dff_all_neuron.shape, Dff_resampled.shape

In [ ]:
recording['ca_times'].shape

resample eye position signal to Display frequency first and interpolate, so we can directly select from the already upsampled Dff signal, otherwise non-continuous signal would need to be entered in the Dff calculation pipeline which is probably only for continuous time signals.

In [ ]:
eye_pos_resampled = interp1d(np.array(camera['fish_embedded_frame_time']).squeeze(), eye_pos.T, kind='nearest')(recording['time_resampled']).T
eye_pos_resampled.shape

In [ ]:
q1_min_left=12
q1_min_right=-4
q3_max_left=4
q3_max_right=-10

# 1st quadrant (defined by lower boundaries)
# 3rd quadrant (define by upper boundaries)
plot_eyepositions(eye_pos_resampled,
                  q1_min_left=q1_min_left,
                  q1_min_right=q1_min_right,
                  q1_width=20,
                  q1_height=25,
                  q3_max_left=q3_max_left,
                  q3_max_right=q3_max_right,
                  q3_widht=20,
                  q3_height=25)

In [ ]:
# select data in quadrants
q1_mask=np.logical_and(
    eye_pos_resampled[:,0] > q1_min_left,
    eye_pos_resampled[:,1] > q1_min_right)
q3_mask=np.logical_and(
    eye_pos_resampled[:,0] < q3_max_left,
    eye_pos_resampled[:,1] < q3_max_right)


# eye positions in Quadrants
# q1=eye_pos_resampled[q1_mask,:]
# q3=eye_pos_resampled[q3_mask,:]
# eye_pos_resampled.shape, q1.shape, q3.shape
q1_mask.shape, q3_mask.shape, q1_mask.sum(), q3_mask.sum()

In [ ]:
recording['cmn_motion_vectors_2d'].shape

In [ ]:
signal_selection = recording['signal_selection']
sample_rate = recording['sample_rate']
radial_bin_edges = recording['radial_bin_edges']
cmn_phase_selection = recording['cmn_phase_selection']
motion_vectors_2d = recording['cmn_motion_vectors_2d']

In [ ]:
ss_q1=signal_selection & q1_mask
ss_q1.sum()

In [ ]:
# masking just with quadrants, without signal
motion_vectors_q1=recording['cmn_motion_vectors_2d'][q1_mask]
motion_vectors_q3=recording['cmn_motion_vectors_2d'][q3_mask]
signal_selection_q1=signal_selection[q1_mask]
signal_selection_q3=signal_selection[q3_mask]
cmn_phase_selection_q1=cmn_phase_selection[q1_mask]
cmn_phase_selection_q3=cmn_phase_selection[q3_mask]

motion_vectors_q1.shape, motion_vectors_q3.shape

In [ ]:
# radial bin etas
q1_rbe_true, q1_rbe_norms_true, q1_rbe_bootstrapped = calculate_reverse_correlations_shm_generalAPI(
    motion_vectors_q1,
    signal_selection_q1,
    cmn_phase_selection_q1,
    sample_rate,
    radial_bin_edges,
    bootstrap_num=1024,
    num_workers=12,)

# radial bin etas
q3_rbe_true, q3_rbe_norms_true, q3_rbe_bootstrapped = calculate_reverse_correlations_shm_generalAPI(
    motion_vectors_q3,
    signal_selection_q3,
    cmn_phase_selection_q3,
    sample_rate,
    radial_bin_edges,
    bootstrap_num=1024,
    num_workers=12,)

# fit optic flow fields to CMN estimations